# 부록: 외부 지식베이스의 표준 ID에 개체를 연결합니다

교안 01에서는 같은 개체인지 판별하고, 교안 02에서는 **프로젝트 내부 표준 ID**를 연결했습니다.  
이 부록에서는 그 개체가 **Wikidata의 어느 항목인지** 찾아 외부 ID를 덧붙입니다.  

**실습의 목표**  

1. 내부 표준 ID와 외부 KB의 항목 ID를 구분합니다.  
2. 이름과 별칭으로 후보를 찾고, 타입과 공식 정보로 하나를 선택합니다.  
3. Neo4j 노드에 `external_id`와 `source_kb`를 저장하고 다시 조회합니다.  
4. 연결할 항목이 없으면 그 사실을 보류로 기록합니다.  

LangChain 프레임워크를 예로 진행합니다. Wikidata 검색과 항목 조회에는 API 키가 필요하지 않습니다.  
Neo4j는 본 교안과 같은 실습용 DB를 사용합니다.  


## 1. 내부 표준 ID와 외부 KB의 항목 ID를 구분합니다

**지식베이스**(KB, Knowledge Base)는 개체와 그 속성, 관계를 항목별로 정리한 자료입니다.  
**개체 연결**(EL, Entity Linking)은 내 자료의 개체가 KB의 어느 항목을 가리키는지 정하는 일입니다.  
원문에 나온 표현을 바로 연결하기도 하고, ER로 한 개체로 묶은 뒤 그 개체를 연결하기도 합니다.  
이 부록은 뒤쪽 방식이며, 교안에서 표준 ID를 부여한 개체를 외부 KB에 연결합니다.  
연결 대상은 내부 KB일 수도 있고 외부 KB일 수도 있습니다. 여기서는 외부 KB인 **Wikidata**를 사용합니다.  

Wikidata는 항목을 `Q`와 숫자로 된 **Q-ID**로, 속성을 `P`와 숫자로 된 **P-ID**로 구분합니다.  
공식 웹사이트 속성은 `P856`이며, 3절에서 이 속성으로 후보를 확인합니다.  
언어별 표기가 달라도 같은 항목이면 Q-ID는 하나입니다.  
LangChain 항목의 이름은 한국어로 `랭체인`, 영어로 `LangChain`이지만 Q-ID는 같습니다.  
KB마다 ID 체계가 달라 ID 문자열만으로는 어디서 받은 값인지 알 수 없으므로, 출처 KB를 함께 기록합니다.  

| 노드 속성 | 역할 | 이번 실습의 값 |
|---|---|---|
| `standard_id` | 프로젝트 안에서 쓰는 개체 ID | `framework:langchain` |
| `external_id` | 외부 KB에서 확인한 항목 ID | `Q117340550` |
| `source_kb` | 외부 ID를 발급한 KB | `wikidata` |

내부 ID를 외부 ID로 덮어쓰지 않습니다. **내부 연결은 유지하고 외부 항목과의 대응을 추가합니다.**  
외부 ID를 기록하는 것과 중복 노드를 `apoc.refactor.mergeNodes`로 합치는 것은 별도 작업입니다.  

[Wikidata의 항목과 식별자](https://www.wikidata.org/wiki/Help:Items)  


## 2. 이름과 별칭으로 Q-ID 후보를 찾습니다

Q-ID를 모르면 먼저 `canonical_name`과 `aliases`로 KB를 검색합니다.  
이름 검색은 **검토할 후보를 좁히는 단계**이며, 검색 결과의 첫 항목을 바로 정답으로 쓰지 않습니다.  
EL에서는 이 단계를 **후보 생성**이라고 부릅니다. 교안 01의 블로킹이 비교할 쌍을 좁힌 것과 같은 역할입니다.  

앞서 ER로 정리한 표준 이름과 별칭은 검색어로 활용할 수 있습니다.  
다만 ER이 끝났다고 외부 KB의 항목까지 자동으로 결정되는 것은 아닙니다.  


#### 연결할 개체와 검색 도구 준비
아래는 LLM 애플리케이션 개발 문서에서 확인한 LangChain 프레임워크의 개체 기록입니다.  
이름뿐 아니라 개체 유형과 공식 사이트도 남겨, 동명이거나 비슷한 이름의 항목을 구분합니다.  


In [ ]:
# requests는 Wikidata의 공개 HTTP API를 호출할 때 사용합니다.
import time
import requests
from pprint import pprint

local_entity = {
    "standard_id": "framework:langchain",
    "canonical_name": "LangChain",
    "aliases": ["랭체인"],
    "entity_type": "SoftwareFramework",
    "context": "Python으로 LLM 애플리케이션을 개발하는 소프트웨어 프레임워크",
    "official_website": "https://langchain.com/",
}
# 공개 API에 요청하는 학습용 클라이언트임을 표시합니다.
headers = {"User-Agent": "EntityLinkingLesson/1.0 (educational Wikidata lookup)"}
pprint(local_entity)


#### 이름과 별칭으로 후보 검색
`wbsearchentities`로 검색어마다 최대 5개를 받고, 중복 Q-ID는 한 번만 남깁니다.  
5절에서 다른 개체로 한 번 더 검색하므로 함수로 만들어 둡니다.  
후보의 **Q-ID, 이름, 설명**을 확인하세요. 결과와 순서는 바뀔 수 있습니다.  
누구나 쓰는 공개 API라 짧은 시간에 요청이 몰리면 `429` 응답이 옵니다.  
그래서 호출 함수가 잠시 기다렸다 다시 요청하며, 그래도 안 되면 안내와 함께 멈춥니다.  


In [ ]:
# 이 부록의 모든 요청이 함께 쓰는 호출 함수입니다.
def get_json(url, params):
    """Wikidata에 요청해 JSON을 돌려주며, 요청이 몰려 429가 오면 기다렸다 다시 시도합니다."""
    for attempt in range(3):
        response = requests.get(url, params=params, headers=headers, timeout=30)
        # 429는 짧은 시간에 요청이 몰렸다는 응답입니다. 여럿이 같이 실행하면 자주 나옵니다.
        if response.status_code == 429:
            # 서버가 Retry-After로 대기 시간을 알려 주면 그 값을 따릅니다.
            wait_seconds = int(response.headers.get("Retry-After", 15 * (attempt + 1)))
            print(f"요청이 몰려 {wait_seconds}초 기다렸다 다시 시도합니다.")
            time.sleep(wait_seconds)
            continue
        # 다른 실패를 그냥 넘기면 빈 결과를 후보 없음으로 잘못 읽게 됩니다.
        response.raise_for_status()
        return response.json()
    raise RuntimeError("Wikidata 요청이 몰려 있습니다. 잠시 뒤 이 셀을 다시 실행하세요.")

def search_candidates(terms, limit=5):
    """검색어마다 후보를 받아 Q-ID를 키로 하는 딕셔너리로 돌려줍니다."""
    found = {}
    for term in terms:
        # language는 먼저 찾을 언어이며, 기본 설정에서는 다른 언어의 표기도 함께 검색됩니다.
        # uselang은 결과의 이름과 설명을 어느 언어로 보여줄지 정합니다.
        result = get_json("https://www.wikidata.org/w/api.php",
                          {"action": "wbsearchentities", "search": term,
                           "language": "ko", "uselang": "ko", "format": "json", "limit": limit})
        for item in result["search"]:
            found[item["id"]] = item
    return found

search_terms = [local_entity["canonical_name"]] + local_entity["aliases"]
candidates_by_id = search_candidates(search_terms)

print("검색어:", search_terms, "/ 중복을 뺀 후보:", len(candidates_by_id))
for qid, item in candidates_by_id.items():
    # 항목마다 채워진 내용이 달라 이름이나 설명이 비어 있을 수 있습니다.
    print(qid, item.get("label", "이름 없음"))
    print("설명:", item.get("description", "설명 없음"))
    print()


## 3. 맥락과 공식 정보를 대조해 항목 하나를 고릅니다

**중의성 해소**는 후보 중 현재 맥락이 가리키는 대상을 선택하는 일입니다.  

| 같은 이름 | 주변 정보 | 구분할 대상 |
|---|---|---|
| Apple | iPhone을 제조한다는 관계 | 회사 |
| Apple | 과일의 영양 정보 | 사과 |

이름만 비슷한 항목도 검색에 함께 딸려 옵니다. 확장 패키지, 강의 자료, 논문이 그런 예입니다.  
그래서 **이름, 타입, 원문과 이웃 관계, 공식 정보를 함께 확인합니다.** 정보가 부족하면 연결을 보류합니다.  

이번 개체는 이름이 비슷한 확장 패키지나 강의 자료가 아니라 **LangChain 프레임워크 본체**입니다.  
후보마다 타입과 공식 사이트를 읽어, 원문에서 확인한 정보와 대조합니다.  


#### 후보마다 타입과 공식 사이트 확인
`wbgetentities`로 후보 전체의 속성을 한 번에 받습니다.  
`P31`은 항목이 어떤 종류인지, `P856`은 공식 웹사이트를 알려 주는 속성입니다.  
타입은 값이 Q-ID로 오므로, 그 Q-ID의 이름을 한 번 더 조회해 사람이 읽을 수 있게 만듭니다.  
이름은 한국어, 영어, 언어를 가리지 않는 공통 표기(`mul`) 순으로 찾습니다.  


In [ ]:
# 후보를 하나씩 조회하지 않고 Q-ID 목록을 한 번의 요청으로 보냅니다.
def get_entities(ids, props):
    """Q-ID 목록의 속성을 한 번에 받아 Q-ID별 딕셔너리로 돌려줍니다."""
    result = get_json("https://www.wikidata.org/w/api.php",
                      {"action": "wbgetentities", "ids": "|".join(ids), "props": props,
                       "languages": "ko|en|mul", "format": "json"})
    return result["entities"]

def claim_values(entity, property_id):
    """그 속성의 값 목록을 돌려주며, 값이 비어 있는 진술은 건너뜁니다."""
    values = []
    for claim in entity["claims"].get(property_id, []):
        value = claim["mainsnak"].get("datavalue")
        if value is not None:
            values.append(value["value"])
    return values

def text_of(entity, field):
    """labels처럼 언어별로 저장된 값을 한국어, 영어, 공통 표기 순으로 돌려줍니다."""
    # 항목마다 채워진 언어가 달라 없는 언어를 바로 꺼내면 KeyError가 납니다.
    # mul은 언어를 가리지 않는 공통 표기로, 한국어와 영어가 모두 없을 때 쓰입니다.
    for language in ["ko", "en", "mul"]:
        if language in entity[field]:
            return entity[field][language]["value"]
    return "값 없음"

candidate_entities = get_entities(list(candidates_by_id), "claims|labels")
# P31의 값은 타입의 Q-ID이므로, 그 Q-ID들의 이름을 한 번 더 받아 옵니다.
type_ids = sorted({value["id"] for entity in candidate_entities.values()
                   for value in claim_values(entity, "P31")})
type_entities = get_entities(type_ids, "labels") if type_ids else {}

for qid, entity in candidate_entities.items():
    type_names = [text_of(type_entities[value["id"]], "labels")
                  for value in claim_values(entity, "P31")]
    print(qid, text_of(entity, "labels"))
    print("  타입:", type_names or "없음")
    print("  공식 사이트:", claim_values(entity, "P856") or "없음")


위 출력을 읽는 방법입니다.  

- 타입이 내 개체와 다른 후보는 먼저 제외합니다. 강의 자료나 논문이 여기에 해당합니다.
- 타입이 같은 후보끼리는 타입만으로 고를 수 없습니다. 확장 패키지도 같은 타입으로 나옵니다.
- 남은 후보는 공식 사이트로 가릅니다. 확장 패키지에는 공식 사이트가 없거나 다른 주소가 적혀 있습니다.

KB의 타입 이름은 내 `entity_type` 표기와 다릅니다.  
LangChain 항목의 타입은 소프트웨어와 파이썬 라이브러리로 나오지만 내 기록은 `SoftwareFramework`입니다.  
두 표기를 잇는 대응표 없이 문자열만 비교하면 맞는 후보도 탈락하므로, 타입은 사람이 읽고 판단하는 근거로 씁니다.  


#### 선택한 항목의 상세 정보 확인
후보 표에서 고른 `Q117340550`의 전체 내용을 받아 마지막으로 확인합니다.  
`P856`의 공식 웹사이트와 내 자료의 사이트를 대조하세요.  


In [ ]:
# 표에서 고른 항목입니다. 사이트 대조 전에는 아직 DB에 저장하지 않습니다.
selected_qid = "Q117340550"
assert selected_qid in candidates_by_id, "검색 결과에서 LangChain 프레임워크 항목을 확인하세요."

# Special:EntityData는 항목 하나의 전체 내용을 JSON으로 내려 주는 주소입니다.
item_json = get_json(f"https://www.wikidata.org/wiki/Special:EntityData/{selected_qid}.json", {})
kb_item = item_json["entities"][selected_qid]

# 앞에서 만든 함수를 그대로 써서 공식 웹사이트 속성의 값을 꺼냅니다.
official_websites = claim_values(kb_item, "P856")

print("Q-ID:", selected_qid)
# 같은 항목이라도 언어마다 이름이 다릅니다. 언어별 이름을 함께 확인합니다.
for language in ["ko", "en"]:
    if language in kb_item["labels"]:
        print(f"항목 이름({language}):", kb_item["labels"][language]["value"])
print("항목 설명:", text_of(kb_item, "descriptions"))
print("KB의 공식 사이트:", official_websites)
print("내 자료의 공식 사이트:", local_entity["official_website"])


[LangChain의 Wikidata 항목](https://www.wikidata.org/wiki/Q117340550)은  
언어 모델 애플리케이션 개발 프레임워크이며, 공식 사이트로 `https://langchain.com/`을 제시합니다.  
타입과 개발 문맥이 내 개체와 어긋나지 않고 공식 사이트도 원문에서 확인한 주소와 같으므로,  
이 실습에서는 **`Q117340550`으로 연결합니다.**  
사이트 문자열이 같다는 검사 하나만으로 모든 개체의 연결을 자동 확정하는 규칙을 만들지는 않습니다.  


## 4. 확인한 외부 ID를 Neo4j 노드에 기록합니다

`external_id`에는 확인한 Q-ID, `source_kb`에는 `wikidata`를 저장합니다.  
연결을 확정했다는 표시로 `link_status`에 `linked`도 함께 남깁니다. 5절의 보류와 구분하기 위해서입니다.  
Cypher의 `$qid`는 Python에서 전달할 값의 자리입니다. ID를 쿼리 문자열에 직접 이어 붙이지 않습니다.  


#### 실습용 Neo4j 연결
본 교안과 같은 `run_cypher`로 쿼리를 실행합니다. 접속 주소와 연결 성공 여부를 확인하세요.  


In [ ]:
# 외부 ID를 기록할 실습용 Neo4j에 연결합니다.
import os
from urllib.parse import urlsplit
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase

# 현재 작업 폴더부터 상위로 올라가 가장 가까운 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()

def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 목록으로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]

# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print("Neo4j 연결 완료. 호스트:", connection_address.hostname, "/ 포트:", connection_address.port)


### 🖐️ 함께 따라하기: LangChain 노드에 Wikidata ID를 붙입니다

#### 내부 표준 ID로 노드 준비

같은 내부 ID의 노드를 만들거나 재사용합니다.  
원래 타입인 `SoftwareFramework`를 라벨로 쓰고, 내부 표준 ID로 LangChain을 구분합니다.  


In [ ]:
# [제공코드] 같은 내부 ID의 노드가 있으면 재사용합니다. 외부 ID는 다음 셀에서 기록합니다.
prepared_nodes = run_cypher("""
// 이름 대신 내부 표준 ID로 이 개체의 노드를 찾거나 만듭니다.
MERGE (n:$($entity_type) {standard_id: $standard_id})
SET n.name = $name, n.aliases = $aliases, n.entity_type = $entity_type
RETURN n.standard_id AS standard_id, n.name AS name
""", standard_id=local_entity["standard_id"],
     name=local_entity["canonical_name"], aliases=local_entity["aliases"],
     entity_type=local_entity["entity_type"])
pprint(prepared_nodes)


#### Q-ID와 KB 이름 저장
- 원래 타입인 `SoftwareFramework` 라벨과 `standard_id`로 노드를 찾아 `external_id = $qid`, `source_kb = 'wikidata'`, `link_status = 'linked'`를 기록하세요.
- `qid=selected_qid`를 전달하고, `name`, `external_id`, `source_kb` 조회 결과를 **external_links**에 담아 출력하세요.


In [ ]:
# (1) run_cypher에 아래 매개변수를 전달하세요.
#     standard_id=local_entity["standard_id"], qid=selected_qid

# (2) MATCH로 해당 standard_id의 SoftwareFramework 노드를 찾으세요.

# (3) SET으로 외부 ID와 출처, 연결 상태를 기록하세요.
#     external_id=$qid, source_kb="wikidata", link_status="linked"

# (4) REMOVE로 이전 보류 사유인 review_reason을 지우세요.

# (5) name, external_id, source_kb를 반환해 external_links에 담고 출력하세요.

# 여기에 코드를 작성하세요.


#### 내부 ID와 외부 ID 보존 확인
LangChain 노드 하나에 `framework:langchain`, `Q117340550`, `wikidata`가 기록되어야 합니다.  
쿼리를 다시 실행해도 같은 내부 ID의 노드가 늘어나지 않아야 합니다.  


In [ ]:
# 쓰기 요청의 반환값과 별도로 DB에 저장된 값을 다시 읽습니다.
saved_links = run_cypher("""
// 같은 내부 ID에 외부 ID가 기록된 상태를 조회합니다.
MATCH (n:SoftwareFramework {standard_id: $standard_id})
RETURN n.standard_id AS standard_id, n.external_id AS external_id, n.source_kb AS source_kb
""", standard_id=local_entity["standard_id"])
pprint(saved_links)

assert len(saved_links) == 1, "내부 ID가 같은 노드가 중복됐는지 확인하세요."
assert saved_links[0]["standard_id"] == local_entity["standard_id"]
assert saved_links[0]["external_id"] == selected_qid
assert saved_links[0]["source_kb"] == "wikidata"
print("내부 ID 보존과 외부 KB 연결 확인 완료")


## 5. 연결할 항목이 없으면 보류로 남깁니다

외부 KB가 모든 개체를 항목으로 갖고 있지는 않습니다.  
KB는 보통 라이브러리 단위까지 항목을 두고, 그 안의 함수 하나까지는 두지 않습니다.  
교안에서 표준 ID를 부여한 `api:seaborn.rugplot`이 그런 경우입니다.  

이때 이름이 비슷한 다른 항목을 골라 붙이면 틀린 연결이 그대로 남습니다.  
연결하지 않기로 한 것도 결과이므로 **왜 연결하지 않았는지를 함께 기록합니다.**  
KB에 항목이 생기거나 검색어를 바꾸면 그 기록을 보고 다시 검토합니다.  


#### 함수 이름과 라이브러리 이름으로 후보 검색
2절에서 만든 `search_candidates`로 두 이름을 각각 검색해, KB가 어느 단위까지 항목을 두는지 확인합니다.  
건수와 순서는 바뀔 수 있으므로 출력을 직접 읽으세요.  


In [ ]:
# 교안에서 표준 ID를 부여한 함수 개체입니다. 외부 KB에 대응 항목이 있는지 확인합니다.
pending_entity = {
    "standard_id": "api:seaborn.rugplot",
    "canonical_name": "rugplot",
    "aliases": ["sns.rugplot"],
    "entity_type": "ApiElement",
    "context": "Seaborn 문서에서 추출한 그래프 함수",
}

# 함수 이름으로 찾은 후보와, 그 함수가 속한 라이브러리 이름으로 찾은 후보를 비교합니다.
function_candidates = search_candidates([pending_entity["canonical_name"]] + pending_entity["aliases"])
library_candidates = search_candidates(["seaborn"])

print("함수 이름 후보:", len(function_candidates))
print("라이브러리 이름 후보:", len(library_candidates))
for qid, item in library_candidates.items():
    print(qid, item.get("label", "이름 없음"), "/", item.get("description", "설명 없음"))


라이브러리 이름으로는 후보가 나오지만, 사람 이름처럼 뜻이 다른 항목이 함께 들어옵니다.  
함수 이름으로는 후보가 나오지 않거나, 나오더라도 이 함수를 가리키는 항목이 아닙니다.  
고를 대상이 없으므로 이 개체는 연결하지 않고, 연결하지 않았다는 사실을 노드에 남깁니다.  


#### 보류 상태로 저장
연결하지 않은 개체도 노드로 남기고 `link_status`와 사유를 기록합니다.  
`external_id`는 비워 두어 확정된 연결과 구분합니다.  


In [ ]:
# 연결을 확정하지 않았으므로 external_id 없이 상태와 사유만 기록합니다.
pending_nodes = run_cypher("""
// 보류한 개체도 같은 라벨과 내부 표준 ID로 남깁니다.
MERGE (n:$($entity_type) {standard_id: $standard_id})
SET n.name = $name, n.entity_type = $entity_type,
    n.link_status = 'review', n.review_reason = $reason
// 이전 외부 연결이 있더라도 현재 판정이 보류이면 확정 ID를 남기지 않습니다.
REMOVE n.external_id, n.source_kb
RETURN n.standard_id AS standard_id, n.link_status AS link_status, n.review_reason AS review_reason
""", standard_id=pending_entity["standard_id"], name=pending_entity["canonical_name"],
     entity_type=pending_entity["entity_type"],
     reason="외부 KB에 이 함수의 항목이 없어 후보를 고르지 못했습니다.")
pprint(pending_nodes)


#### 연결과 보류를 한 번에 조회
부록에서 만든 노드를 모두 읽어 상태를 비교합니다.  
연결한 개체에는 `external_id`가 있고, 보류한 개체에는 사유가 남아 있어야 합니다.  


In [ ]:
# 두 개체의 상태를 한 번에 읽어 연결과 보류가 구분되는지 확인합니다.
link_states = run_cypher("""
// 현재 자료의 두 표준 ID로 조회합니다. 공통 라벨은 추가하지 않습니다.
MATCH (n)
WHERE n.standard_id IN $standard_ids
RETURN n.standard_id AS standard_id, labels(n) AS labels, n.link_status AS link_status,
       n.external_id AS external_id, n.review_reason AS review_reason
ORDER BY n.standard_id
""", standard_ids=[local_entity["standard_id"], pending_entity["standard_id"]])
pprint(link_states)

expected_ids = {local_entity["standard_id"], pending_entity["standard_id"]}
assert len(link_states) == 2, "같은 표준 ID의 노드가 중복됐는지 확인하세요."
assert {row["standard_id"] for row in link_states} == expected_ids
states_by_id = {row["standard_id"]: row for row in link_states}
linked = states_by_id[local_entity["standard_id"]]
pending = states_by_id[pending_entity["standard_id"]]
assert linked["labels"] == [local_entity["entity_type"]], "원래 타입을 라벨로 사용하세요."
assert pending["labels"] == [pending_entity["entity_type"]], "보류한 개체도 원래 타입을 유지하세요."
assert linked["link_status"] == "linked", "연결한 개체의 상태를 확인하세요."
assert linked["external_id"] == selected_qid, "연결한 개체의 external_id를 확인하세요."
assert pending["link_status"] == "review", "보류한 개체의 상태를 확인하세요."
assert pending["external_id"] is None, "보류한 개체에는 external_id를 넣지 않습니다."
print("연결 1건과 보류 1건이 상태로 구분되어 저장되었습니다.")


**후보 검색은 이름으로, 최종 선택은 맥락으로 합니다.**  
ER로 정리한 이름과 별칭은 검색을 돕고, EL은 그 개체가 외부 KB의 어떤 항목인지 연결합니다.  
근거가 부족하면 연결하지 않고 보류로 남기며, ID만 저장하지 않고 어느 KB에서 온 ID인지 함께 기록하세요.  

[Wikidata 검색 API](https://www.wikidata.org/w/api.php?action=help&modules=wbsearchentities)  
[Wikidata 항목 JSON 조회](https://www.wikidata.org/wiki/Wikidata:Data_access)  
